# ModernBERT domain classifier (v4)

Fixes vs v3:
1. `compute_metrics` no longer shadows `labels=` into `f1_score`; removed meaningless `pos_label=1` for multi-class.
2. `push_to_hub=False` by default (avoids auth errors when the HF login cell is not run). Re-enable after logging in.
3. Dynamic padding via `DataCollatorWithPadding` instead of padding inside `dataset.map`.

**Security note:** the v3 notebook had a real HF token pasted into a commented-out cell. Even commented, it is committed to history — **revoke that token on huggingface.co and generate a new one** before pushing.

In [1]:
# Install Hugging Face libraries
%pip install --upgrade \
  "datasets==3.1.0" \
  "accelerate==1.2.1" \
  "hf-transfer==0.1.8"

# ModernBERT is not yet available in an official release, so we need to install it from github
%pip install "git+https://github.com/huggingface/transformers.git@6e0515e99c39444caae39472ee1b2fd76ece32f1" --upgrade

/Users/alexanderquispe/Documents/GitHub/Data-Science-Python/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
/Users/alexanderquispe/Documents/GitHub/Data-Science-Python/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Uncomment and paste a fresh token if you want to push to the Hub.
# Do NOT commit a real token — generate one at https://huggingface.co/settings/tokens
# from huggingface_hub import login
# login(token="hf_xxx_REPLACE_ME", add_to_git_credential=True)

In [2]:
from datasets import load_dataset

dataset_id = "argilla/synthetic-domain-text-classification"

train_dataset = load_dataset(dataset_id, split="train")

split_dataset = train_dataset.train_test_split(test_size=0.1)
split_dataset["train"][0]

README.md:   0%|          | 0.00/3.24k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  336kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

{'text': 'The film industry has been a central pillar of entertainment for over a century now. This medium allows directors and screenwriters to craft elaborate stories that captivate audiences around the world. Over time, it has evolved from silent movies in black and white to the high-definition color experiences we enjoy today. Behind every hit movie is a collaborative effort between actors, producers, set designers, composers, and cinematographers. For instance, the directorial style of Quentin Tarantino can be noted for its distinctive use of nonlinear storytelling, anachronistic dialogue, and stylized violence. Additionally, music plays an integral role in shaping the mood and emotional impact of films; it is often used to evoke specific feelings from viewers as they follow the journey of characters through different narratives.',
 'label': 21}

In [14]:
split_dataset["train"][0]

{'text': 'The film industry has been a central pillar of entertainment for over a century now. This medium allows directors and screenwriters to craft elaborate stories that captivate audiences around the world. Over time, it has evolved from silent movies in black and white to the high-definition color experiences we enjoy today. Behind every hit movie is a collaborative effort between actors, producers, set designers, composers, and cinematographers. For instance, the directorial style of Quentin Tarantino can be noted for its distinctive use of nonlinear storytelling, anachronistic dialogue, and stylized violence. Additionally, music plays an integral role in shaping the mood and emotional impact of films; it is often used to evoke specific feelings from viewers as they follow the journey of characters through different narratives.',
 'labels': 21}

In [11]:
from transformers import AutoTokenizer, DataCollatorWithPadding

model_id = "answerdotai/ModernBERT-base"

tokenizer = AutoTokenizer.from_pretrained(model_id)

# Tokenize without padding here; pad per-batch via DataCollatorWithPadding at training time.
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True)

if "label" in split_dataset["train"].features.keys():
    split_dataset = split_dataset.rename_column("label", "labels")  # match Trainer
tokenized_dataset = split_dataset.map(tokenize, batched=True, remove_columns=["text"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

tokenized_dataset["train"].features.keys()

config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

dict_keys(['labels', 'input_ids', 'attention_mask'])

In [13]:
tokenized_dataset["train"][0]

{'labels': 21,
 'input_ids': [50281,
  510,
  3085,
  4491,
  556,
  644,
  247,
  4275,
  43035,
  273,
  14608,
  323,
  689,
  247,
  5331,
  1024,
  15,
  831,
  4646,
  4483,
  17970,
  285,
  3601,
  34782,
  281,
  11072,
  21184,
  6281,
  326,
  3403,
  400,
  366,
  23886,
  1475,
  253,
  1533,
  15,
  6061,
  673,
  13,
  352,
  556,
  16323,
  432,
  11561,
  11321,
  275,
  2806,
  285,
  3168,
  281,
  253,
  1029,
  14,
  28692,
  3295,
  8450,
  359,
  4264,
  3063,
  15,
  33341,
  1046,
  4352,
  6440,
  310,
  247,
  27549,
  3434,
  875,
  14142,
  13,
  18014,
  13,
  873,
  22507,
  13,
  43039,
  13,
  285,
  15573,
  14359,
  42679,
  15,
  1198,
  4227,
  13,
  253,
  6423,
  451,
  3740,
  273,
  3277,
  41319,
  308,
  5746,
  2610,
  476,
  320,
  4879,
  323,
  697,
  21488,
  897,
  273,
  14561,
  41651,
  13,
  271,
  607,
  1406,
  2531,
  17414,
  13,
  285,
  17521,
  1025,
  7217,
  15,
  9157,
  13,
  3440,
  7120,
  271,
  9909,
  2554,
  275,
  2

In [21]:
from transformers import AutoModelForSequenceClassification

model_id = "answerdotai/ModernBERT-base"

labels = tokenized_dataset["train"].features["labels"].names
num_labels = len(labels)
label2id = {label: str(i) for i, label in enumerate(labels)}
id2label = {str(i): label for i, label in enumerate(labels)}
id2label

{'0': 'business-and-industrial',
 '1': 'books-and-literature',
 '2': 'beauty-and-fitness',
 '3': 'autos-and-vehicles',
 '4': 'people-and-society',
 '5': 'sports',
 '6': 'shopping',
 '7': 'online-communities',
 '8': 'pets-and-animals',
 '9': 'internet-and-telecom',
 '10': 'home-and-garden',
 '11': 'adult',
 '12': 'science',
 '13': 'food-and-drink',
 '14': 'real-estate',
 '15': 'news',
 '16': 'jobs-and-education',
 '17': 'health',
 '18': 'hobbies-and-leisure',
 '19': 'games',
 '20': 'computers-and-electronics',
 '21': 'arts-and-entertainment',
 '22': 'travel-and-transportation',
 '23': 'finance',
 '24': 'law-and-government',
 '25': 'sensitive-subjects'}

In [22]:
from transformers import AutoModelForSequenceClassification

model_id = "answerdotai/ModernBERT-base"

labels = tokenized_dataset["train"].features["labels"].names
num_labels = len(labels)
label2id = {label: str(i) for i, label in enumerate(labels)}
id2label = {str(i): label for i, label in enumerate(labels)}

model = AutoModelForSequenceClassification.from_pretrained(
    model_id, num_labels=num_labels, label2id=label2id, id2label=id2label,
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  599MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [24]:
import numpy as np
from sklearn.metrics import f1_score

# Fixed: no more `labels=labels` shadowing, no meaningless `pos_label=1`.
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    score = f1_score(labels, predictions, average="weighted")
    return {"f1": float(score)}

In [25]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="ModernBERT-domain-classifier",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,
    num_train_epochs=5,
    bf16=True,
    optim="adamw_torch_fused",
    logging_strategy="steps",
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    # Set push_to_hub=True only after running the login cell with a valid token.
    push_to_hub=False,
    hub_strategy="every_save",
    gradient_checkpointing=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [26]:
trainer.train()
tokenizer.save_pretrained("ModernBERT-domain-classifier")
trainer.create_model_card()
# Only call push_to_hub() after enabling push_to_hub=True above and logging in.
# trainer.push_to_hub()

/Users/alexanderquispe/Documents/GitHub/Data-Science-Python/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,F1
1,1.465201,0.414868,0.868571
2,0.405705,0.426380,0.885224
3,0.122650,0.292143,0.901465
4,0.053689,0.248082,0.927708
5,0.030488,0.215685,0.941994


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/alexanderquispe/Documents/GitHub/Data-Science-Python/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/alexanderquispe/Documents/GitHub/Data-Science-Python/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/alexanderquispe/Documents/GitHub/Data-Science-Python/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [27]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

CUDA available: False


In [29]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch

model_id = "argilla/ModernBERT-domain-classifier"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(
    model_id, torch_dtype=torch.float16
).to("mps")

classifier = pipeline(
    task="text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0,
)

classifier("Smoking is bad for your health.")

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

[{'label': 'health', 'score': 0.6796403527259827}]

In [30]:
samples = [
    "Smoking is bad for your health.",
    "Google is one of the largest tech companies.",
    "New agricultural tools have boosted rice production.",
]
classifier(samples, batch_size=8)

[{'label': 'health', 'score': 0.6792837977409363},
 {'label': 'online-communities', 'score': 0.2721675634384155},
 {'label': 'science', 'score': 0.2653372883796692}]